# Huấn luyện & So sánh Model — Phân loại tổn thương da (HAM10000)

Train **deep learning** trên tập **train/val/test đã chia sẵn** (chia theo `lesion_id`, không rò rỉ), so sánh `efficientnet_b0` / `resnet50` / `mobilenet_v3_large`, xuất `production.pt` đúng định dạng hệ thống serving.

**Cách dùng**: Add dataset HAM10000 → bật GPU → chỉnh vùng CONFIG nếu cần → Run.
Muốn chạy nhanh: đặt `SUBSET_PER_CLASS = 300`, `EPOCHS = 3`, hoặc `ARCHS = ["efficientnet_b0"]`.

In [ ]:
import os
import json
import glob
import random
import os.path as osp

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Subset
from torchvision import datasets, models, transforms
from sklearn.metrics import accuracy_score, f1_score, recall_score


def find_data_dir():
    cands = [
        "/kaggle/input/datasets/htrnthi/ham10000-images/data",
        "/kaggle/input/ham10000-images/data",
        "/kaggle/working/data",
        "../data",
    ]
    cands += sorted(glob.glob("/kaggle/input/**/data", recursive=True))
    for p in cands:
        if osp.isdir(osp.join(p, "train")) and osp.isdir(osp.join(p, "val")):
            return p
    return cands[0]


DATA_DIR = find_data_dir()
TRAIN_DIR = "train"
VAL_DIR = "val"
TEST_DIR = "test"

ARCHS = ["efficientnet_b0", "resnet50", "mobilenet_v3_large"]
IMG_SIZE = 224
BATCH_SIZE = 32
EPOCHS = 5
LR = 1e-3
SEED = 42
SUBSET_PER_CLASS = None
FREEZE_BACKBONE = True
PRETRAINED = True
NUM_WORKERS = 2
PROD_METRIC = "macro_f1"
OUTPUT_DIR = "/kaggle/working"

random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
torch.cuda.manual_seed_all(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
USE_AMP = DEVICE.type == "cuda"
print("DATA_DIR:", DATA_DIR, "| device:", DEVICE, "| amp:", USE_AMP)

IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

train_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomVerticalFlip(),
    transforms.RandomRotation(20),
    transforms.ColorJitter(0.1, 0.1, 0.1),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])
eval_tf = transforms.Compose([
    transforms.Resize((IMG_SIZE, IMG_SIZE)),
    transforms.ToTensor(),
    transforms.Normalize(IMAGENET_MEAN, IMAGENET_STD),
])


def subset_per_class(dataset, n_per_class, seed):
    if not n_per_class:
        return dataset
    rng = random.Random(seed)
    by_class = {}
    for idx, (_, label) in enumerate(dataset.samples):
        by_class.setdefault(label, []).append(idx)
    keep = []
    for label, idxs in by_class.items():
        rng.shuffle(idxs)
        keep.extend(idxs[:n_per_class])
    keep.sort()
    return Subset(dataset, keep)


train_full = datasets.ImageFolder(osp.join(DATA_DIR, TRAIN_DIR), transform=train_tf)
val_set = datasets.ImageFolder(osp.join(DATA_DIR, VAL_DIR), transform=eval_tf)
test_set = datasets.ImageFolder(osp.join(DATA_DIR, TEST_DIR), transform=eval_tf)

CLASSES = train_full.classes
NUM_CLASSES = len(CLASSES)
MEL_INDEX = CLASSES.index("mel") if "mel" in CLASSES else 0
train_set = subset_per_class(train_full, SUBSET_PER_CLASS, SEED)
print("classes:", CLASSES)
print("train:", len(train_set), "| val:", len(val_set), "| test:", len(test_set))


def class_weights(dataset):
    if isinstance(dataset, Subset):
        labels = [dataset.dataset.samples[i][1] for i in dataset.indices]
    else:
        labels = [label for _, label in dataset.samples]
    counts = np.bincount(labels, minlength=NUM_CLASSES).astype(np.float64)
    counts[counts == 0] = 1.0
    weights = counts.sum() / (NUM_CLASSES * counts)
    return torch.tensor(weights, dtype=torch.float32)


CLASS_WEIGHTS = class_weights(train_set).to(DEVICE)


def make_loader(dataset, shuffle):
    return DataLoader(dataset, batch_size=BATCH_SIZE, shuffle=shuffle, num_workers=NUM_WORKERS, pin_memory=USE_AMP)


train_loader = make_loader(train_set, True)
val_loader = make_loader(val_set, False)
test_loader = make_loader(test_set, False)


def build_model(name):
    weights = "DEFAULT" if PRETRAINED else None
    if name == "efficientnet_b0":
        m = models.efficientnet_b0(weights=weights)
        m.classifier[1] = nn.Linear(m.classifier[1].in_features, NUM_CLASSES)
        backbone = m.features
    elif name == "resnet50":
        m = models.resnet50(weights=weights)
        m.fc = nn.Linear(m.fc.in_features, NUM_CLASSES)
        backbone = nn.Sequential(m.conv1, m.bn1, m.layer1, m.layer2, m.layer3, m.layer4)
    elif name == "mobilenet_v3_large":
        m = models.mobilenet_v3_large(weights=weights)
        m.classifier[3] = nn.Linear(m.classifier[3].in_features, NUM_CLASSES)
        backbone = m.features
    else:
        raise ValueError(name)
    if FREEZE_BACKBONE:
        for p in backbone.parameters():
            p.requires_grad = False
    return m


def train_one_epoch(model, loader, optimizer, criterion, scaler):
    model.train()
    running = 0.0
    for images, labels in loader:
        images, labels = images.to(DEVICE), labels.to(DEVICE)
        optimizer.zero_grad()
        if USE_AMP:
            with torch.cuda.amp.autocast():
                loss = criterion(model(images), labels)
            scaler.scale(loss).backward()
            scaler.step(optimizer)
            scaler.update()
        else:
            loss = criterion(model(images), labels)
            loss.backward()
            optimizer.step()
        running += loss.item() * images.size(0)
    return running / len(loader.dataset)


def evaluate(model, loader):
    model.eval()
    y_true, y_pred = [], []
    with torch.no_grad():
        for images, labels in loader:
            preds = model(images.to(DEVICE)).argmax(1).cpu().numpy()
            y_pred.extend(preds.tolist())
            y_true.extend(labels.numpy().tolist())
    return {
        "accuracy": round(float(accuracy_score(y_true, y_pred)), 4),
        "macro_f1": round(float(f1_score(y_true, y_pred, average="macro", zero_division=0)), 4),
        "melanoma_recall": round(float(recall_score(y_true, y_pred, labels=[MEL_INDEX], average="macro", zero_division=0)), 4),
    }


def run_experiment(name):
    print("=" * 48, "\nARCH:", name)
    model = build_model(name).to(DEVICE)
    criterion = nn.CrossEntropyLoss(weight=CLASS_WEIGHTS)
    optimizer = torch.optim.Adam([p for p in model.parameters() if p.requires_grad], lr=LR)
    scaler = torch.cuda.amp.GradScaler(enabled=USE_AMP)
    for epoch in range(1, EPOCHS + 1):
        loss = train_one_epoch(model, train_loader, optimizer, criterion, scaler)
        print(f"epoch {epoch}/{EPOCHS} loss={loss:.4f} val={evaluate(model, val_loader)}")
    val_metrics = evaluate(model, val_loader)
    test_metrics = evaluate(model, test_loader)
    state_dict = {k: v.cpu() for k, v in model.state_dict().items()}
    print("val:", val_metrics, "| test:", test_metrics)
    return {"arch": name, "val": val_metrics, "test": test_metrics, "state_dict": state_dict}


results = [run_experiment(name) for name in ARCHS]

rows = []
for r in results:
    rows.append({
        "arch": r["arch"],
        "val_acc": r["val"]["accuracy"], "val_macro_f1": r["val"]["macro_f1"], "val_mel_recall": r["val"]["melanoma_recall"],
        "test_acc": r["test"]["accuracy"], "test_macro_f1": r["test"]["macro_f1"], "test_mel_recall": r["test"]["melanoma_recall"],
    })
table = pd.DataFrame(rows).sort_values("val_macro_f1", ascending=False).reset_index(drop=True)
print(table.to_string(index=False))

best = max(results, key=lambda r: r["val"][PROD_METRIC])
checkpoint = {
    "classes": CLASSES,
    "model_name": best["arch"],
    "version": "kaggle_v1",
    "data_version": "kaggle_split",
    "image_size": IMG_SIZE,
    "state_dict": best["state_dict"],
    "val_metrics": best["val"],
    "test_metrics": best["test"],
}
out_path = osp.join(OUTPUT_DIR, "production.pt")
torch.save(checkpoint, out_path)
with open(osp.join(OUTPUT_DIR, "results.json"), "w") as handle:
    json.dump([{k: r[k] for k in ("arch", "val", "test")} for r in results], handle, indent=2)
print("saved:", out_path, "| best:", best["arch"], "| val:", best["val"])

## Cắm vào hệ thống MLOps

1. Tải `production.pt` từ Output của Kaggle.
2. Chép vào `code/backend/app/models/production.pt`.
3. Gọi `POST /admin/reload-model` (kèm `x-admin-token`) hoặc đăng ký vào MLflow rồi promote sang **Production**.

> Checkpoint đúng định dạng `ModelService` (`classes`, `model_name`, `version`, `image_size`, `state_dict`) → cắm thẳng, Grad-CAM chạy được vì là CNN.